In [0]:
from typing import Any

In [0]:
import logging

from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.functions import (
    col,
    concat_ws,
    current_date,
    current_timestamp,
    from_utc_timestamp,
    lit,
    regexp_replace,
    sha2,
    split,
    when,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
    sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
    sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()

In [0]:
def load_curated_estatement_extfile(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args):
        date_filter = f"businessdate = {run_date!r}"

        cluster_columns = ["BusinessDate"]
        curated_table_path = f"{curated_zone}.estatements"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(spark, curated_table_path, cluster_columns, schema)
        else:
            logging.info("Table already exists.")
        
        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(spark, curated_table_path, run_date, "extfile")
        else:
            logging.info("Table does not exist, no need to delete entries for business date!")

        # ================== extfile_EStatementStatistics (main fact) ==================

        estatement_extra_filters = None

        if estatement_extra_filters is not None:
            estatement_extra_filters.append(date_filter)
            estatement_final_filters = " AND ".join(estatement_extra_filters)
        else:
            estatement_final_filters = date_filter

        estatement_selected_fields = [
            "StagingKey",
            "StagingDateInserted",
            "Filename",
            "ErrorCode",
            "ErrorMessage",
        ]

        estatement = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "extfile_estatementstatistics",
            estatement_final_filters,
            estatement_selected_fields,
        )

        # ================== Parse Filename into components ==================
        # Format: Company#PortfolioProfitsUnit#ExposureKey#CDI#Month#Year.pdf

        estatement = estatement.withColumn(
            "Filename_NoExt", regexp_replace(col("Filename"), "\\.pdf$", "")
        )
              
        split_col = split(col("Filename_NoExt"), "#")
        estatement = (
            estatement
            .withColumn("Company", split_col.getItem(0))
            .withColumn("PortfolioProfitsUnit", split_col.getItem(1))
            .withColumn("FilenameExposureKey", regexp_replace(split_col.getItem(2), "[^0-9]", ""))
            .withColumn("FilenameCDI", split_col.getItem(3))
            .withColumn("StatementProductionMonth", split_col.getItem(4))
            .withColumn("StatementProductionYear", split_col.getItem(5))
            .drop("Filename_NoExt")
        )
    
        # ================== profits_customer (for TaxRegistrationNumber) ==================

        customer_extra_filters = None

        if customer_extra_filters is not None:
            customer_extra_filters.append(date_filter)
            customer_final_filters = " AND ".join(customer_extra_filters)
        else:
            customer_final_filters = date_filter

        customer_selected_fields = [
            "CDI",
            "CUST_ID",
            "CUST_STATUS",
            "tax_number",
            "E_MAIL",
        ]

        customer = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "profits_customer",
            customer_final_filters,
            customer_selected_fields,
        )

        # ================== profits_ProfitsAccCustLnk (account-customer link) ==================

        acccustlnk_extra_filters = None

        if acccustlnk_extra_filters is not None:
            acccustlnk_extra_filters.append(date_filter)
            acccustlnk_final_filters = " AND ".join(acccustlnk_extra_filters)
        else:
            acccustlnk_final_filters = date_filter

        acccustlnk_selected_fields = [
            "FK_CUSTOMER",
            "FK_ACCOUNT",
            "RECIPIENT_FLAG",
            "ESTATEMENT_FLG",
        ]

        acccustlnk = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "profits_ProfitsAccCustLnk",
            acccustlnk_final_filters,
            acccustlnk_selected_fields,
        )

        # ================== primary.portfolio (for PortfolioName) ==================

        portfolio_selected_fields = ["PortfolioName", "unit_code_profits"]

        portfolio = load_landed_portfolio_table(
            spark,
            curated_zone,  # type: ignore
            "portfolio",
            date_filter,
            portfolio_selected_fields,
        )

        # ========================== Joins ==========================

        # INNER JOIN profits_customer on CDI
        base = estatement.join(customer, estatement.FilenameCDI == customer.CDI, "inner")

        # LEFT JOIN primary.portfolio on unit_code_profits
        portfolio_cond = (
            F.expr("try_cast(PortfolioProfitsUnit as int)") == portfolio.unit_code_profits
        )
        base = base.join(portfolio, portfolio_cond, "left")

        # LEFT JOIN profits_ProfitsAccCustLnk
        acccustlnk_cond = (
            (acccustlnk.FK_CUSTOMER == customer.CUST_ID)
            # & (F.substring(acccustlnk.FK_ACCOUNT, 4, 100) == estatement.FilenameExposureKey)
        )
        base = base.join(acccustlnk, acccustlnk_cond, "left")

        target_table = base.filter(col("RECIPIENT_FLAG") == "1")

        # ========================== Primary Key ==========================

        primary_key_fields = [
            estatement.FilenameExposureKey,
            estatement.FilenameCDI,
            estatement.StagingKey,
            lit("extfile"),
        ]

        target_table = (
            target_table
            .withColumn("EStatementId", sha2(concat_ws(";", *primary_key_fields), 256))
            .withColumn("EStatementKey", concat_ws(";", estatement.FilenameExposureKey, estatement.FilenameCDI, estatement.StagingKey))
        )

        # ========================== Select Final Columns ==========================

        result_df = target_table.select(
            target_table.EStatementId.cast("string").alias("EStatementId"),
            lit("extfile").cast("string").alias("SourceSystem"),
            estatement.PortfolioProfitsUnit.cast("string").alias("PortfolioKey"),
            portfolio.PortfolioName.cast("string").alias("PortfolioName"),
            from_utc_timestamp(current_timestamp(), "Europe/Athens").alias("IngestionTimestamp"),
            target_table.EStatementKey.cast("string").alias("EStatementKey"),
            estatement.FilenameExposureKey.cast("string").alias("ExposureKeyWithoutPrefix"),
            estatement.FilenameCDI.cast("string").alias("CDI"),
            customer.tax_number.cast("string").alias("TaxRegistrationNumber"),
            customer.E_MAIL.cast("string").alias("Email"),
            acccustlnk.ESTATEMENT_FLG.cast("string").alias("OptIn"),
            when(customer.CUST_STATUS == "4", True).otherwise(False).alias("DeceasedFlag"),
            estatement.Company.cast("string").alias("Company"),
            estatement.StatementProductionMonth.cast("string").alias("StatementProductionMonth"),
            estatement.StatementProductionYear.cast("string").alias("StatementProductionYear"),
            estatement.StagingKey.cast("string").alias("StagingKey"),
            estatement.StagingDateInserted.cast("timestamp").alias("StagingDateInserted"),
            estatement.Filename.cast("string").alias("Filename"),
            estatement.ErrorCode.cast("string").alias("ErrorCode"),
            estatement.ErrorMessage.cast("string").alias("ErrorMessage"),
            lit(run_date).cast("date").alias("BusinessDate"),
        ).distinct()

        # ========================== Append ==========================

        result_df.write.format("delta").clusterBy("BusinessDate").mode("append").saveAsTable(
            curated_table_path
        )

    else:
        null_args = [
            name for name, val in zip(
                ["curated_zone", "catalog_landed", "reference_zone", "run_date", "schema", "env_prefix"], args
            ) if val is None
        ]
        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("estatements curated load complete.")

In [0]:
load_curated_estatement_extfile(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)